In [1]:
import numpy as np

In [8]:
import pandas as pd
import re

# --- Config ---
filename = "C:\\Users\\Aishwary Sharma\\Downloads\\Unique_metabolites_416.csv"   # change to your actual CSV file path

# --- Load the data ---
try:
    df = pd.read_csv(filename, encoding='utf-8')
except UnicodeDecodeError:
    print("⚠️ UTF-8 decoding failed, trying latin1...")
    df = pd.read_csv(filename, encoding='latin1')

# --- Regex to detect special/non-ASCII characters ---
special_char_pattern = re.compile(r"[^\x20-\x7E]")

# --- Scan all cells for special characters ---
print("🔍 Scanning for special characters...\n")
found_any = False

for col in df.columns:
    for idx, val in df[col].astype(str).items():
        if special_char_pattern.search(val):
            print(f"Row {idx}, Column '{col}': {val}")
            found_any = True

if not found_any:
    print("✅ No special characters found.")
else:
    print("\n✅ Scan complete — special characters detected above.")


🔍 Scanning for special characters...

✅ No special characters found.


In [7]:
df

,classification,Metabolites,Notation
0,lipid_class,?-Galactosylceramide,?-D-GalCer
1,lipid_class,Cardiolipin,CL
2,lipid_class,Chylomicron-remnants(CE),Chylomicron-remnants(CE)
3,lipid_class,Digalactosylceramide(Gal2Cer),Gal2Cer
4,lipid_class,free-cholesterol(fc),FC
...,...,...,...
410,nonlipid,Phosphoethanolamine (PETA),PETA
411,nonlipid,Succinate,Succinate
412,nonlipid,UDP-alpha-D-galactose,UDP-Gal
413,nonlipid,UDP-galactose(UDP-Gal),UDP-Gal


In [34]:
import os
import pandas as pd

dir_path = "D:/Raylab/LiMeNEx_Network/src/sbmlData/pathwayTfsModified"

final_df = pd.DataFrame()

for file in os.listdir(dir_path):
    if file.endswith(".csv"):
        file_path = os.path.join(dir_path, file)
        temp_df = pd.read_csv(file_path, dtype=str)
        final_df = pd.concat([final_df, temp_df], ignore_index=True)


In [35]:
# Clean up strings across all columns
for col in final_df.select_dtypes(include='object').columns:
    final_df[col] = final_df[col].astype(str).str.strip().str.replace('\u00A0', ' ', regex=False)


In [37]:
len(final_df)

73505

In [38]:
import pandas as pd
import numpy as np

# --- Step 1: Read and normalize ---
# df = pd.read_csv("your_file.csv", dtype=str)  # read all as string to prevent float conversion

# Clean whitespace and normalize PMIDs
pmid_cols = ["Chea", "Signor", "Trrust"]
for col in pmid_cols:
    final_df[col] = (
        final_df[col]
        .astype(str)
        .str.strip()
        .replace(["nan", "None"], "", regex=False)
        .str.replace(r"\.0$", "", regex=True)  # remove trailing .0
    )

# --- Step 2: Group and merge duplicates ---
def merge_pmids(group):
    """
    For each (TF, TargetGene, Tissue) group:
    - If any row has a PMID in Chea/Signor/Trrust, keep those
    - Else keep first occurrence
    """
    # Keep all rows that have at least one PMID
    with_pmid = group[group[pmid_cols].apply(lambda x: any(x.notna() & (x != "")), axis=1)]
    if not with_pmid.empty:
        # If multiple with PMIDs exist, keep the one with most PMIDs filled
        with_pmid["pmid_count"] = with_pmid[pmid_cols].apply(lambda x: (x != "").sum(), axis=1)
        best_row = with_pmid.sort_values("pmid_count", ascending=False).iloc[0]
        return best_row.drop("pmid_count")
    else:
        return group.iloc[0]

cleaned_df = final_df.groupby(["TF", "TargetGene", "Tissue"], dropna=False, as_index=False).apply(merge_pmids).reset_index(drop=True)

print(f"✅ Cleaned dataset has {len(cleaned_df)} unique rows.")


✅ Cleaned dataset has 66082 unique rows.


C:\Temp\ipykernel_24904\592646567.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cleaned_df = final_df.groupby(["TF", "TargetGene", "Tissue"], dropna=False, as_index=False).apply(merge_pmids).reset_index(drop=True)


In [40]:
cleaned_df.to_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", index=False)

In [10]:
import pandas as pd
df = pd.read_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", dtype=str)

In [11]:
df.head()

,TF,TargetGene,Tissue,Experiment,Chea,Signor,Trrust
0,125DVD3,ABCA1,colon,CEBPB IP | 125DVD3,NaN,NaN,NaN
1,125DVD3,ABCG1,colon,"CEBPB IP | 125DVD3, CDX2 IP | 125DVD3",NaN,NaN,NaN
2,125DVD3,ABHD3,colon,"CDX2 IP | 125DVD3, CEBPB IP | 125DVD3",NaN,NaN,NaN
3,125DVD3,ACAA1,colon,"CEBPB IP | 125DVD3, CDX2 IP | 125DVD3",NaN,NaN,NaN
4,125DVD3,ACACA,colon,CEBPB IP | 125DVD3,NaN,NaN,NaN


In [ ]:
targetGene = df['TargetGene'].unique().tolist()
len(targetGene)


AttributeError: 'list' object has no attribute 'tolist'

In [7]:
import pickle
with open("targetGene.pkl", "wb") as f:
    pickle.dump(targetGene, f)


In [1]:
import pandas as pd
df = pd.read_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", dtype=str)

In [2]:
mapping = {}


for index, row in df.iterrows():
    tf = row['TF']
    target_gene = row['TargetGene']
    
    if target_gene not in mapping:
        mapping[target_gene] = [tf]
    else:
        if tf not in mapping[target_gene]:
            mapping[target_gene].append(tf)

In [4]:
import json

with open("D:/Raylab/LiMeNEx_Network/src/sbmlData/tf_targetgene_mapping.json", "w") as f:
    json.dump(mapping, f)

Removing Tfs and Enzymatic Genes

In [25]:
import json
import pandas as pd
import os

mapping_file = 'D:/Raylab/LiMeNEx_Network/src/sbmlData/tf_targetgene_mapping_cleaned.json'
not_Tf_genes = ["FUT2", "HSD17B8", "PPT1", "AKR1C8","GK3","COX1","COX2"]

with open(mapping_file, 'r') as f:
    tf_target_mapping = json.load(f) 

enzymatic_genes = set(tf_target_mapping.keys())
print(len(tf_target_mapping.keys()))
enzymatic_genes.update(set(not_Tf_genes))

final_tfs = set()
count = 0
for tfs in tf_target_mapping.values():
    count+=len(tfs)
    final_tfs.update(set(tfs))
          

print(len(final_tfs))
print(count)
# print()

### load dataframe
# df = pd.read_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", dtype=str)


323
345
28184


In [22]:
import pickle
with open("targetGene.pkl", "wb") as f:
    pickle.dump(list(enzymatic_genes), f)

In [6]:
#first filter out enzymatic genes from TargetGene column
final_df = df[df['TargetGene'].isin(enzymatic_genes)]
final_df = final_df[final_df['TF'].isin(final_tfs)]

# remaining_df = df[~final_df]

In [16]:
final_df.shape

(61984, 7)

In [12]:
remaining_df = df[~df.index.isin(final_df.index)]

In [13]:
remaining_df.shape

(4097, 7)

In [ ]:
not_enzymatic_genes = set()
not_valid_tfs = set()

for index, row in remaining_df.iterrows():
    tfs = row['TF']
    tf_list = [tf.strip() for tf in tfs.split(';')]
    
    target_gene = row['TargetGene']
    
    if target_gene not in enzymatic_genes:
        # print(f"Target Gene: {target_gene} is not an enzymatic gene.")
        not_enzymatic_genes.add(target_gene)
        continue
    
    for tf in tf_list:
        # print(f"TF: {tf}, Target Gene: {target_gene}")
        if tf not in final_tfs:
            not_valid_tfs.add(tf)
            continue
        
        tissue = row['Tissue']
        
        temp_df = final_df[(final_df['TF'] == tf) & (final_df['TargetGene'] == target_gene) & (final_df['Tissue'] == tissue)]
        if temp_df.empty:
            final_row = {
                'TF': tf,
                'TargetGene': target_gene,
                'Tissue': tissue,
                'Experiment' : row['Experiment'],
                'Chea': row['Chea'],
                'Signor': row['Signor'],
                'Trrust': row['Trrust']
            }
            final_df = pd.concat([final_df, pd.DataFrame([final_row],index=[index])], ignore_index=True)
        else:
            final_row = {}
            for col in ['Experiment', 'Chea', 'Signor', 'Trrust']:
                existing_value = temp_df.iloc[0][col]
                new_value = row[col]
                
                if pd.isna(existing_value) or existing_value == '':
                    final_row[col] = new_value
                elif pd.isna(new_value) or new_value == '':
                    final_row[col] = existing_value
                else:
                    if col == 'Experiment':
                        existing_Cell_lines = set(line.strip() for line in str(existing_value).split(','))
                        new_Cell_lines = set(line.strip() for line in str(new_value).split(','))
                        combined_Cell_lines = existing_Cell_lines.union(new_Cell_lines)
                        final_row[col] = ', '.join(sorted(combined_Cell_lines))
                    else:
                        existing_pmids = set(pmid.strip() for pmid in str(existing_value).split(';'))
                        new_pmids = set(pmid.strip() for pmid in str(new_value).split(';'))
                        combined_pmids = existing_pmids.union(new_pmids)
                        final_row[col] = ';'.join(sorted(combined_pmids))
        
            for col in ['Experiment', 'Chea', 'Signor', 'Trrust']:
                final_df.loc[temp_df.index, col] = final_row[col]

In [18]:
final_df.shape

(62007, 7)

In [19]:
not_enzymatic_genes

{'ACOT8'}

In [20]:
not_valid_tfs

{'125DVD3',
 '17BE2',
 '22RHC',
 '4HT',
 'AMYLB',
 'ATRA',
 'BAY082',
 'BICALU',
 'CAMPTO',
 'CEP/P300',
 'COMPE',
 'CPDA',
 'DEX',
 'DHT',
 'EFR2',
 'FORSK',
 'GW965',
 'H9',
 'H9RET',
 'H9RETR',
 'INS',
 'JQ1',
 'NROB2',
 'NUTL3',
 'P4',
 'PGE2',
 'PPRAD',
 'PRAV',
 'R1881',
 'R5020',
 'RO306',
 'ROSI',
 'TEFB'}

In [21]:
final_df.to_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/final_tf_targetgene_tissue_groups.csv", index=False)

In [14]:
import pandas as pd

file_path = "D:/Raylab/LiMeNEx_Network/src/sbmlData/final_tf_targetgene_tissue_groups.csv"
found = set()
count = {}

df = pd.read_csv(file_path, dtype={'Experiment': str, 'Trrust': str, 'Signor': str, 'Chea': str})
for index,row in df.iterrows():

    geneKey = row['TF'] + row['TargetGene'] + row['Tissue']
    if geneKey not in found:
        found.add(geneKey)
    else:
        print(index)
        print("Duplicate found:", geneKey)
        continue
    
    isExperiment = True if pd.notna(row["Experiment"]) and row["Experiment"].strip() != "" else False
    isTrrust = True if pd.notna(row["Trrust"]) and row["Trrust"].strip() != "" else False
    isSignor = True if pd.notna(row["Signor"]) and row["Signor"].strip() != "" else False
    isChea = True if pd.notna(row["Chea"]) and row["Chea"].strip() != "" else False
        

    dbKey = ""
    if isExperiment:
        dbKey += "Experiment;"
    if isTrrust:
        dbKey += "Trrust;"
    if isSignor:
        dbKey += "Signor;"
    if isChea:
        dbKey += "Chea;"

    dbKey = dbKey[:-1]
    if dbKey not in count:
        count[dbKey] = 1
    else:
        count[dbKey] += 1

In [15]:
df.shape

(62006, 7)

In [16]:
len(found)

62006

In [17]:
print(count)

{'Signor': 111, 'Trrust': 506, 'Trrust;Signor': 45, 'Trrust;Chea': 7, 'Chea': 1551, 'Trrust;Signor;Chea': 4, 'Experiment': 59429, 'Experiment;Chea': 273, 'Experiment;Trrust': 66, 'Signor;Chea': 1, 'Experiment;Trrust;Signor': 6, 'Experiment;Signor': 5, 'Experiment;Trrust;Chea': 2}


In [ ]:
import pandas as pd

file_path = "D:/Raylab/LiMeNEx_Network/src/sbmlData/final_tf_targetgene_tissue_groups.csv"
found = set()
count = {}

df = pd.read_csv(file_path, dtype={'Experiment': str, 'Trrust': str, 'Signor': str, 'Chea': str})
for index,row in df.iterrows():

    geneKey = row['TF'] + row['TargetGene'] + row['Tissue']
    if geneKey not in found:
        found.add(geneKey)
    else:
        print(index)
        print("Duplicate found:", geneKey)
        continue
    
    isExperiment = True if pd.notna(row["Experiment"]) and row["Experiment"].strip() != "" else False
    isTrrust = True if pd.notna(row["Trrust"]) and row["Trrust"].strip() != "" else False
    isSignor = True if pd.notna(row["Signor"]) and row["Signor"].strip() != "" else False
    isChea = True if pd.notna(row["Chea"]) and row["Chea"].strip() != "" else False
        

    dbKey = ""
    if isExperiment:
        dbKey += "Experiment;"
    if isTrrust:
        dbKey += "Trrust;"
    if isSignor:
        dbKey += "Signor;"
    if isChea:
        dbKey += "Chea;"

    dbKey = dbKey[:-1]
    if dbKey not in count:
        count[dbKey] = 1
    else:
        count[dbKey] += 1

In [1]:
print("Hi")

Hi


In [20]:
import pickle
    
with open("D:/Raylab/LiMeNEx_Network/src/sbmlData/targetGene.pkl", "rb") as f:
    enzymaticGene1 = pickle.load(f)
    
with open("D:/Raylab/LiMeNEx_Network/src/sbmlData/targetGene2.pkl", "rb") as f:
    enzymaticGene2 = pickle.load(f)

In [21]:
len(set(enzymaticGene1))

330

In [22]:
len(set(enzymaticGene2))

330

In [23]:
set(enzymaticGene2) - set(enzymaticGene1)

set()

In [24]:
set(enzymaticGene2) - set(enzymaticGene1)

set()

In [9]:
import pandas as pd

df = pd.read_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/final_tf_targetgene_tissue_groups_FINAL.csv")

def ps_confidence(ps_freq):
    if ps_freq <= 3:
        return 'low'
    elif ps_freq <= 6:
        return 'medium'
    else:
        return 'high'

def tissue_confidence(tissue_freq):
    if tissue_freq <= 3:
        return 'low'
    elif tissue_freq <= 7:
        return 'medium'
    else:
        return 'high'

df["PS_Confidence"] = df["PS_frequency"].apply(ps_confidence)
df["Tissue_Confidence"] = df["Tissue_frequency"].apply(tissue_confidence)

# df.to_csv("output_file.csv", index=False)
df["SPP_Pubmed"] = pd.to_numeric(df["SPP_Pubmed"], errors="coerce").astype("Int64")

In [10]:
df[:20]

,TF,TargetGene,Tissue,Chea,Signor,Trrust,Experiment,SPP_Pubmed,Pair,PS_frequency,Tissue_frequency,PS_Confidence,Tissue_Confidence
0,AR,A4GALT,prostate,NaN,NaN,NaN,WT AR OE + FOXA1 OE | FOXA1 ChIP-Seq,<NA>,AR -> A4GALT,1,1,low,low
1,ATF2,A4GALT,leukocytes,NaN,NaN,NaN,ATF2 IP - GM12878 cells,24076218,ATF2 -> A4GALT,1,1,low,low
2,BATF,A4GALT,leukocytes,NaN,NaN,NaN,BATF IP - GM12878 cells,24076218,BATF -> A4GALT,1,1,low,low
3,BCL11A,A4GALT,leukocytes,NaN,NaN,NaN,BCL11A IP - GM12878 cells,24076218,BCL11A -> A4GALT,1,1,low,low
4,BHLHE40,A4GALT,leukocytes,NaN,NaN,NaN,BHLHE40 IP - GM12878 cells,24076218,BHLHE40 -> A4GALT,1,1,low,low
5,CEBPA,A4GALT,leukocytes,NaN,NaN,NaN,CEBPA IP,24198249,CEBPA -> A4GALT,1,1,low,low
6,CEBPB,A4GALT,bone,NaN,NaN,NaN,CEBPB IP,26111340,CEBPB -> A4GALT,6,6,medium,medium
7,CEBPB,A4GALT,leukocytes,NaN,NaN,NaN,"CEBPB IP | oxLDL - Rep 208, CEBPB IP",<NA>,CEBPB -> A4GALT,6,6,medium,medium
8,CEBPB,A4GALT,liver,NaN,NaN,NaN,CEBPB ChIP-Seq - HepG2,<NA>,CEBPB -> A4GALT,6,6,medium,medium
9,CEBPB,A4GALT,lung,NaN,NaN,NaN,"CEBPB ChIP-Seq - A549, CEBPB ChIP-Seq - IMR90",<NA>,CEBPB -> A4GALT,6,6,medium,medium


In [11]:
df.to_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/final_tf_targetgene_tissue_groups_FINAL_FINAL.csv.csv", index=False)

In [21]:
# Group by Pair and check for multiple confidence scores
pair_ps = df.groupby("Pair")["PS_Confidence"].nunique()
pair_tissue = df.groupby("Pair")["Tissue_Confidence"].nunique()

# Filter pairs with more than 1 unique confidence score
multi_ps = pair_ps[pair_ps > 1]
multi_tissue = pair_tissue[pair_tissue > 1]

print(f"Pairs with >1 PS_Confidence: {len(multi_ps)}")
print(multi_ps)

print(f"\nPairs with >1 Tissue_Confidence: {len(multi_tissue)}")
print(multi_tissue)

# Combined - pairs that have multiple in either column
multi_either = df.groupby("Pair").filter(
    lambda x: x["PS_Confidence"].nunique() > 1 or x["Tissue_Confidence"].nunique() > 1
)
print(f"\nPairs with >1 confidence in either column:\n{multi_either[['Pair','PS_Confidence','Tissue_Confidence']].drop_duplicates()}")

Pairs with >1 PS_Confidence: 0
Series([], Name: PS_Confidence, dtype: int64)

Pairs with >1 Tissue_Confidence: 0
Series([], Name: Tissue_Confidence, dtype: int64)

Pairs with >1 confidence in either column:
Empty DataFrame
Columns: [Pair, PS_Confidence, Tissue_Confidence]
Index: []
